## Job Failure Scraper — Runbook

- Use widgets to configure the time window, workspace host, page limit, log table, and token source.
- Ensure `job_failure_scraper.py` and `args.py` are available in this Repo so imports work.


In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

import job_failure_scraper as jfs

# Defaults
default_host = "https://" + spark.conf.get("spark.databricks.workspaceUrl")
default_start = "2025-09-23 08:00:00"
default_end = "2025-09-23 10:59:59"
default_limit = "26"
default_log_table = "your_catalog.your_schema.job_failure_log"

# Create widgets if missing (defaults show example formats)
for name, value, label in [
  ("start", default_start, "Start time (UTC) e.g. 2025-09-23 08:00:00"),
  ("end", default_end, "End time (UTC) e.g. 2025-09-23 10:59:59"),
  ("host", default_host, "Workspace Host e.g. https://<your-workspace>.azuredatabricks.net"),
  ("limit", default_limit, "API Page Limit (max 26) e.g. 26"),
  ("log_table", default_log_table, "Log table (catalog.schema.table) e.g. catalog.schema.job_failure_log"),
  ("token", "<paste_PAT_here>", "PAT (optional; leave blank to use secret)"),
  ("secret_scope", "<secret_scope>", "Secret Scope (optional)"),
  ("secret_key", "<secret_key>", "Secret Key (optional)"),
]:
  try:
    dbutils.widgets.get(name)
  except Exception:
    dbutils.widgets.text(name, value, label)

# Read widgets
start = dbutils.widgets.get("start").strip()
end = dbutils.widgets.get("end").strip()
host = dbutils.widgets.get("host").strip() or default_host
limit = dbutils.widgets.get("limit").strip() or default_limit
log_table = dbutils.widgets.get("log_table").strip()
user_token = dbutils.widgets.get("token").strip()
secret_scope = dbutils.widgets.get("secret_scope").strip()
secret_key = dbutils.widgets.get("secret_key").strip()

# Resolve token: prefer token widget; otherwise secrets
if user_token and user_token != "<paste_PAT_here>":
  pat = user_token
elif secret_scope and secret_scope != "<secret_scope>" and secret_key and secret_key != "<secret_key>":
  pat = dbutils.secrets.get(scope=secret_scope, key=secret_key)
else:
  raise ValueError("No PAT provided. Fill the 'token' widget or provide 'secret_scope' and 'secret_key'.")

# Build args
args = [
  "--start", start,
  "--end", end,
  "--host", host,
  "--token", pat,
  "--limit", limit,
]
if log_table:
  args += ["--log-table", log_table]

jfs.main(args)
